# 02 – Seleção e limpeza

**Objetivo:** aplicar os tratamentos definidos no diagnóstico do notebook 01, por meio de funções
documentadas em `src/funcoes.py`. Cada etapa mostra o estado dos dados antes e depois do tratamento.
Ao final, o dataset limpo é salvo para as análises do notebook 03.

In [7]:
import sys
sys.path.append("..")

%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np

from src.funcoes import (
    carregar_dados, selecionar_colunas, renomear_colunas,
    converter_para_numerico, preencher_faltantes,
)

pd.set_option("display.max_columns", None)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
df_bruto = carregar_dados("../data/spotify-2023.csv")

Dados carregados: 953 linhas e 24 colunas.


## Etapa 1 – Seleção de colunas
Mantemos apenas as 19 colunas relacionadas ao Spotify (ver planejamento). As colunas de Apple Music,
Deezer e Shazam ficam fora do escopo; com isso, também saem `in_deezer_playlists` e
`in_shazam_charts`, que tinham tipos incorretos e valores faltantes.

In [4]:
colunas_spotify = [
    "track_name", "artist(s)_name", "artist_count",
    "released_year", "released_month", "released_day",
    "streams", "in_spotify_playlists", "in_spotify_charts",
    "bpm", "key", "mode",
    "danceability_%", "valence_%", "energy_%", "acousticness_%",
    "instrumentalness_%", "liveness_%", "speechiness_%",
]

df = selecionar_colunas(df_bruto, colunas_spotify)
print(f"Antes: {df_bruto.shape} | Depois: {df.shape}")

Antes: (953, 24) | Depois: (953, 19)


In [5]:
try:
    selecionar_colunas(df_bruto, ["track_name", "coluna_que_nao_existe"])
    print("FALHOU: a função deveria ter levantado um erro.")
except KeyError as erro:
    print(f"Teste OK – erro capturado: {erro}")

Teste OK – erro capturado: "Colunas não encontradas no DataFrame: ['coluna_que_nao_existe']"


## Etapa 2 – Renomeação de colunas
Removemos símbolos dos nomes (`(s)`, `%`), que dificultam a escrita do código.
As demais colunas já estão em um formato adequado e mantêm o nome original,
o que preserva a correspondência com a documentação do Kaggle.

In [6]:
novos_nomes = {
    "artist(s)_name": "artist_name",
    "danceability_%": "danceability",
    "valence_%": "valence",
    "energy_%": "energy",
    "acousticness_%": "acousticness",
    "instrumentalness_%": "instrumentalness",
    "liveness_%": "liveness",
    "speechiness_%": "speechiness",
}

df = renomear_colunas(df, novos_nomes)
df.columns.tolist()

['track_name',
 'artist_name',
 'artist_count',
 'released_year',
 'released_month',
 'released_day',
 'streams',
 'in_spotify_playlists',
 'in_spotify_charts',
 'bpm',
 'key',
 'mode',
 'danceability',
 'valence',
 'energy',
 'acousticness',
 'instrumentalness',
 'liveness',
 'speechiness']

## Etapa 3 – Conversão de `streams` para número
`streams` foi lida como texto por causa de um único valor corrompido (linha 574, "Love Grows").
Esse valor vira `NaN`; a linha é mantida, pois as demais colunas estão íntegras (remoção caso a caso).

In [8]:
print(f"Tipo antes: {df['streams'].dtype}")
print(f"Linha 574 antes: {df.loc[574, 'streams']}\n")

df = converter_para_numerico(df, "streams")

print(f"\nTipo depois: {df['streams'].dtype}")
print(f"Linha 574 depois: {df.loc[574, 'streams']}")

Tipo antes: str
Linha 574 antes: BPM110KeyAModeMajorDanceability53Valence75Energy69Acousticness7Instrumentalness0Liveness17Speechiness3

Aviso: 1 valor(es) de 'streams' não eram numéricos e viraram NaN.

Tipo depois: float64
Linha 574 depois: nan


In [9]:
df["streams"].describe()

count    9.520000e+02
mean     5.141374e+08
std      5.668569e+08
min      2.762000e+03
25%      1.416362e+08
50%      2.905309e+08
75%      6.738690e+08
max      3.703895e+09
Name: streams, dtype: float64

In [10]:
teste = pd.DataFrame({"x": ["10", "20", "abc", None]})
resultado = converter_para_numerico(teste, "x")
print(resultado)

assert resultado["x"].isna().sum() == 2, "Deveria haver 2 NaN"
assert resultado["x"].iloc[0] == 10, "O primeiro valor deveria ser 10"
print("Testes OK")

Aviso: 1 valor(es) de 'x' não eram numéricos e viraram NaN.
      x
0  10.0
1  20.0
2   NaN
3   NaN
Testes OK


## Etapa 4 – Preenchimento dos faltantes de `key`
Os 95 faltantes correspondem à nota C (C nunca aparece na coluna; verificação externa de uma amostra
aleatória de 10 músicas: 10 de 10 em C). Preenchemos com "C", registrando essa suposição.

In [11]:
antes = df["key"].value_counts(dropna=False)

df = preencher_faltantes(df, "key", "C")

depois = df["key"].value_counts(dropna=False)
pd.DataFrame({"antes": antes, "depois": depois}).fillna(0).astype(int)

95 valor(es) faltante(s) em 'key' preenchido(s) com 'C'.


,antes,depois
key,,
A,75,75
A#,57,57
B,81,81
C,0,95
C#,120,120
D,81,81
D#,33,33
E,62,62
F,89,89


In [12]:
teste = pd.DataFrame({"x": ["A", None, "B", None]})
resultado = preencher_faltantes(teste, "x", "Z")

assert resultado["x"].isna().sum() == 0, "Não deveria sobrar NaN"
assert (resultado["x"] == "Z").sum() == 2, "Deveriam ser 2 valores 'Z'"
assert teste["x"].isna().sum() == 2, "A tabela original não deveria ser alterada"
print("Testes OK")

2 valor(es) faltante(s) em 'x' preenchido(s) com 'Z'.
Testes OK
